# 03 — Model Training v2 (IEEE-CIS)

**Değişiklikler (v2):**
- Dual-Branch mimari: VAE ham feature üzerinde çalışıyor (GAT embedding değil)
- NeighborLoader ile mini-batch training (~10x hız)
- Temporal + Identity edge'leri aktif
- Differential LR (VAE catastrophic forgetting çözümü)
- XGBoost baseline eklendi
- Daha büyük model kapasitesi (gat_hidden=128, gat_out=64)

In [5]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
import time
from torch_geometric.loader import NeighborLoader

# Device
if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
print(f'Device: {device}')

# Config
with open('../configs/default.yaml') as f:
    cfg = yaml.safe_load(f)
print('\nModel:', {k: v for k, v in cfg['model'].items()})
print('Training:', {k: v for k, v in cfg['training'].items()})

Device: mps

Model: {'gat_hidden': 128, 'gat_out': 64, 'gat_heads': 4, 'gat_layers': 2, 'vae_latent': 32, 'vae_hidden': 128, 'dropout': 0.3}
Training: {'lr': 0.001, 'lr_vae': 0.0001, 'lr_classifier': 0.002, 'weight_decay': 0.0001, 'epochs': 50, 'patience': 10, 'batch_size': 2048, 'num_neighbors': [10, 5], 'lambda1': 0.3, 'lambda2': 0.01, 'focal_gamma': 2.0, 'focal_alpha': 0.75}


## 1. Graph Oluşturma (Temporal + Identity Edge'ler Dahil)

In [6]:
from src.graph.builder import build_hetero_graph

# Raw veri
print('Veri yukleniyor...')
train_txn = pd.read_csv('../data/raw/ieee_cis/train_transaction.csv')
train_id = pd.read_csv('../data/raw/ieee_cis/train_identity.csv')
df = train_txn.merge(train_id, on='TransactionID', how='left')
print(f'Satir: {len(df):,}, Kolon: {len(df.columns)}')

# Kucuk veri ile test icin (RAM sorunu cozulunce kaldir)
df = df.sample(frac=0.2, random_state=42).reset_index(drop=True)
print(f'Kucultulmus veri: {len(df):,} satir (frac=0.2)')

# Graph olustur
print('\nGraph olusturuluyor...')
data = build_hetero_graph(df, dataset='ieee_cis')

# Label bilgisi
y = data['transaction'].y
n_fraud = y.sum().item()
n_total = len(y)
print(f'\nFraud: {n_fraud:,} / {n_total:,} ({100*n_fraud/n_total:.2f}%)')

# Kaydet
os.makedirs('../data/processed/ieee_cis', exist_ok=True)
torch.save(data, '../data/processed/ieee_cis/hetero_graph_v3.pt')
print('Graph kaydedildi: hetero_graph_v3.pt')

# Raw features'i ayrica sakla (VAE icin)
raw_txn_features = data['transaction'].x.clone()
print(f'Raw txn features shape: {raw_txn_features.shape}')


Veri yukleniyor...
Satir: 590,540, Kolon: 434
Kucultulmus veri: 118,108 satir (frac=0.2)

Graph olusturuluyor...


/Users/berkkilic/Documents/aiprojects/graduation-p/src/graph/builder.py:47: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["merchant_key"] = df["addr1"].astype(str) + "_" + df["ProductCD"].astype(str)


  Temporal edges: 30,196 (window=3600s, max_neighbors=5)
  Identity edges: 49,968 (cols=['DeviceType', 'DeviceInfo', 'id_30', 'id_31'])
Graph built:
  Transactions: 118,108 (388 features)
  Accounts: 8,161 (5 features)
  Merchants: 344 (3 features)
  Edges: 118,108 (initiates + reverse)
  Edges: 105,142 (paid_to + reverse)
  Edges: 30,196 (temporal: followed_by + preceded_by)
  Edges: 49,968 (identity: shares_identity + reverse)

Fraud: 4,242 / 118,108 (3.59%)
Graph kaydedildi: hetero_graph_v3.pt
Raw txn features shape: torch.Size([118108, 388])


## 2. Train/Val/Test Split (Stratified)

In [7]:
from sklearn.model_selection import train_test_split

y_np = y.numpy()
indices = np.arange(n_total)

# Stratified split: %70 train, %15 val, %15 test
train_idx, temp_idx = train_test_split(
    indices, test_size=0.3, stratify=y_np, random_state=42
)
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.5, stratify=y_np[temp_idx], random_state=42
)

train_mask = torch.zeros(n_total, dtype=torch.bool)
val_mask = torch.zeros(n_total, dtype=torch.bool)
test_mask = torch.zeros(n_total, dtype=torch.bool)
train_mask[train_idx] = True
val_mask[val_idx] = True
test_mask[test_idx] = True

data['transaction'].train_mask = train_mask
data['transaction'].val_mask = val_mask
data['transaction'].test_mask = test_mask

print(f'Train: {train_mask.sum().item():,} (fraud: {y[train_mask].sum().item():,})')
print(f'Val:   {val_mask.sum().item():,} (fraud: {y[val_mask].sum().item():,})')
print(f'Test:  {test_mask.sum().item():,} (fraud: {y[test_mask].sum().item():,})')

Train: 82,675 (fraud: 2,969)
Val:   17,716 (fraud: 636)
Test:  17,717 (fraud: 637)


## 3. XGBoost Baseline (Tabular — Graph Kullanmadan)

Literatürde IEEE-CIS'de XGBoost AUC~0.94-0.96 alır. Bu, graph-based modelin referans noktasıdır.

In [8]:
from xgboost import XGBClassifier
from src.evaluation.metrics import compute_metrics, print_report

# XGBoost ham feature'larla çalışır (graph yok)
X_train = raw_txn_features[train_mask].numpy()
X_val = raw_txn_features[val_mask].numpy()
X_test = raw_txn_features[test_mask].numpy()
y_train = y[train_mask].numpy()
y_val = y[val_mask].numpy()
y_test = y[test_mask].numpy()

# Fraud oranı: pos/neg weight
fraud_ratio = y_train.sum() / (len(y_train) - y_train.sum())

print('XGBoost eğitiliyor...')
t0 = time.time()
xgb = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=1/fraud_ratio,
    eval_metric='aucpr',
    early_stopping_rounds=20,
    tree_method='hist',       # hızlı histogram-based
    n_jobs=-1,
    random_state=42,
    verbosity=0,
)
xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
xgb_time = time.time() - t0
print(f'XGBoost eğitim süresi: {xgb_time:.1f}s')

# Test sonuçları
xgb_scores = xgb.predict_proba(X_test)[:, 1]
print('\n=== XGBoost Baseline ===')
xgb_metrics = compute_metrics(y_test, xgb_scores)
print_report(y_test, xgb_scores)

XGBoost eğitiliyor...
XGBoost eğitim süresi: 5.8s

=== XGBoost Baseline ===

--- Fraud Detection Metrics (threshold=0.803) ---
  threshold           : 0.8029
  f1_fraud            : 0.5722
  recall_fraud        : 0.4851
  precision_fraud     : 0.6975
  roc_auc             : 0.9070
  avg_precision       : 0.5740

  Confusion Matrix:
[[16946   134]
 [  328   309]]


## 4. GCN Baseline (Homojen Graph)

In [9]:
from src.models.baselines import GCNBaseline
from src.training.losses import focal_loss
from collections import defaultdict

# Homojen graph: aynı account'a bağlı transaction'ları birbirine bağla
acct_txn_edges = data['account', 'initiates', 'transaction'].edge_index
txn_x = data['transaction'].x

acct_to_txns = defaultdict(list)
for a, t in zip(acct_txn_edges[0].tolist(), acct_txn_edges[1].tolist()):
    acct_to_txns[a].append(t)

src_list, dst_list = [], []
for txns in acct_to_txns.values():
    if len(txns) > 1:
        for i in range(len(txns)):
            for j in range(i+1, min(i+5, len(txns))):
                src_list.extend([txns[i], txns[j]])
                dst_list.extend([txns[j], txns[i]])

homo_edge_index = torch.tensor([src_list, dst_list], dtype=torch.long)
print(f'Homojen graph: {txn_x.shape[0]:,} node, {homo_edge_index.shape[1]:,} edge')

# GCN eğitimi
gcn = GCNBaseline(in_channels=txn_x.shape[1], hidden=64, out=32).to(device)
opt_gcn = torch.optim.Adam(gcn.parameters(), lr=0.001, weight_decay=1e-4)

txn_x_dev = txn_x.to(device)
homo_edges_dev = homo_edge_index.to(device)
y_dev = y.float().to(device)
train_mask_dev = train_mask.to(device)
val_mask_dev = val_mask.to(device)

best_val = float('inf')
patience_cnt = 0

print('GCN eğitiliyor...')
t0 = time.time()
for epoch in range(1, 101):
    gcn.train()
    opt_gcn.zero_grad()
    logits = gcn(txn_x_dev, homo_edges_dev)
    loss = focal_loss(logits[train_mask_dev], y_dev[train_mask_dev])
    loss.backward()
    opt_gcn.step()

    gcn.eval()
    with torch.no_grad():
        vl = focal_loss(gcn(txn_x_dev, homo_edges_dev)[val_mask_dev], y_dev[val_mask_dev]).item()
    if vl < best_val:
        best_val = vl; patience_cnt = 0
        best_gcn = {k: v.cpu().clone() for k, v in gcn.state_dict().items()}
    else:
        patience_cnt += 1
        if patience_cnt >= 15:
            print(f'Early stopping epoch {epoch}')
            break
    if epoch % 25 == 0:
        print(f'  Epoch {epoch:3d} | train: {loss.item():.4f} | val: {vl:.4f}')

gcn.load_state_dict(best_gcn)
gcn_time = time.time() - t0
print(f'GCN eğitim süresi: {gcn_time:.1f}s')

# GCN test
gcn.eval()
with torch.no_grad():
    gcn_scores = torch.sigmoid(gcn(txn_x_dev, homo_edges_dev)).cpu().numpy()
gcn_test_scores = gcn_scores[test_mask.numpy()]

print('\n=== GCN Baseline ===')
gcn_metrics = compute_metrics(y_test, gcn_test_scores)
print_report(y_test, gcn_test_scores)

Homojen graph: 118,108 node, 828,320 edge
GCN eğitiliyor...
  Epoch  25 | train: 0.0209 | val: 0.0207
  Epoch  50 | train: 0.0201 | val: 0.0199
  Epoch  75 | train: 0.0197 | val: 0.0196
  Epoch 100 | train: 0.0195 | val: 0.0194
GCN eğitim süresi: 50.1s

=== GCN Baseline ===

--- Fraud Detection Metrics (threshold=0.419) ---
  threshold           : 0.4187
  f1_fraud            : 0.2909
  recall_fraud        : 0.3265
  precision_fraud     : 0.2623
  roc_auc             : 0.7830
  avg_precision       : 0.2109

  Confusion Matrix:
[[16495   585]
 [  429   208]]


## 5. Hybrid GAT+VAE (Dual-Branch, NeighborLoader)

**Mimari:**
- Branch A: HeteroGAT → h_t (graph-based embeddings)
- Branch B: VAE(raw_features) → reconstruction error (anomaly score)
- Fusion: classifier([h_t || recon_err]) → fraud score

**Eğitim:**
- Phase 1: VAE pre-training (sadece normal txn, graph yok → DataLoader ile hızlı)
- Phase 2: End-to-end NeighborLoader ile mini-batch + differential LR

In [10]:
from src.models.hybrid_model import HybridGATVAE
from src.training.losses import focal_loss, reconstruction_loss, kl_divergence

# Model oluştur
in_channels = {ntype: data[ntype].x.shape[1] for ntype in data.node_types}
raw_txn_dim = data['transaction'].x.shape[1]

model = HybridGATVAE(
    metadata=data.metadata(),
    in_channels=in_channels,
    raw_txn_dim=raw_txn_dim,
    gat_hidden=cfg['model']['gat_hidden'],
    gat_out=cfg['model']['gat_out'],
    gat_heads=cfg['model']['gat_heads'],
    gat_layers=cfg['model']['gat_layers'],
    vae_latent=cfg['model']['vae_latent'],
    vae_hidden=cfg['model']['vae_hidden'],
    dropout=cfg['model']['dropout'],
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f'Model parametreleri: {total_params:,}')
print(f'Feature dims: {in_channels}')
print(f'Raw txn dim: {raw_txn_dim}')

Model parametreleri: 1,766,791
Feature dims: {'account': 5, 'merchant': 3, 'transaction': 388}
Raw txn dim: 388


### Phase 1: VAE Pre-training
VAE'yi sadece **normal** (non-fraud) transaction'ların ham feature'ları üzerinde eğitiyoruz.
Graph kullanmıyor — bu yüzden basit DataLoader ile çok hızlı.

In [11]:
from torch.utils.data import DataLoader, TensorDataset

# Sadece normal (non-fraud) train transaction'ların raw feature'ları
normal_train_mask = train_mask & (y == 0)
normal_features = raw_txn_features[normal_train_mask]
print(f'Normal train örnekleri: {len(normal_features):,}')

# VAE parametrelerini aç, geri kalanını dondur
for p in model.gat.parameters(): p.requires_grad = False
for p in model.classifier.parameters(): p.requires_grad = False
for p in model.recon_bn.parameters(): p.requires_grad = False
for p in model.vae.parameters(): p.requires_grad = True

vae_optimizer = torch.optim.Adam(model.vae.parameters(), lr=0.001, weight_decay=1e-4)
vae_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(vae_optimizer, patience=5, factor=0.5)

# DataLoader (hızlı, graph yok)
vae_dataset = TensorDataset(normal_features)
vae_loader = DataLoader(vae_dataset, batch_size=4096, shuffle=True)

print('\n' + '═' * 60)
print('PHASE 1: VAE Pre-training (normal features only)')
print('═' * 60)

phase1_history = []
t0 = time.time()

for epoch in range(1, 31):  # 30 epoch
    model.train()
    epoch_recon, epoch_kl, n_batches = 0, 0, 0

    for (batch_x,) in vae_loader:
        batch_x = batch_x.to(device)
        vae_optimizer.zero_grad()

        x_recon, mu, logvar = model.vae(batch_x)
        l_recon = reconstruction_loss(batch_x, x_recon)
        l_kl = kl_divergence(mu, logvar)
        loss = l_recon + 0.01 * l_kl

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.vae.parameters(), max_norm=1.0)
        vae_optimizer.step()

        epoch_recon += l_recon.item()
        epoch_kl += l_kl.item()
        n_batches += 1

    avg_recon = epoch_recon / n_batches
    avg_kl = epoch_kl / n_batches
    vae_scheduler.step(avg_recon)
    phase1_history.append({'epoch': epoch, 'recon': avg_recon, 'kl': avg_kl})

    if epoch % 10 == 0:
        print(f'  Epoch {epoch:3d} | recon: {avg_recon:.4f} | kl: {avg_kl:.4f}')

p1_time = time.time() - t0
print(f'Phase 1 tamamlandı ({p1_time:.1f}s). Final recon: {phase1_history[-1]["recon"]:.4f}')

# Tüm parametreleri aç
for p in model.parameters(): p.requires_grad = True

Normal train örnekleri: 79,706

════════════════════════════════════════════════════════════
PHASE 1: VAE Pre-training (normal features only)
════════════════════════════════════════════════════════════
  Epoch  10 | recon: 0.2281 | kl: 1.5875
  Epoch  20 | recon: 0.1684 | kl: 1.6586
  Epoch  30 | recon: 0.1434 | kl: 1.6491
Phase 1 tamamlandı (9.9s). Final recon: 0.1434


### Phase 2: End-to-End Training (NeighborLoader + Differential LR)

- GAT + VAE + Classifier birlikte eğitiliyor
- NeighborLoader: her batch'te 2048 transaction + komşuları sample edilir
- VAE lr = 0.1x (catastrophic forgetting önleme)
- Classifier lr = 2x (daha hızlı öğrensin)

In [12]:
# ─── NeighborLoader'lar ───
batch_size = cfg['training']['batch_size']
num_neighbors = cfg['training']['num_neighbors']

train_loader = NeighborLoader(
      data,
      num_neighbors=[5, 3],   # [10, 5] yerine
      batch_size=4096,        # 2048 yerine
      input_nodes=('transaction', train_mask),
      shuffle=True,
)

val_loader = NeighborLoader(
    data,
    num_neighbors=num_neighbors,
    batch_size=batch_size * 2,  # eval daha büyük batch olabilir
    input_nodes=('transaction', val_mask),
    shuffle=False,
)

print(f'Train loader: {len(train_loader)} batches (batch_size={batch_size})')
print(f'Val loader: {len(val_loader)} batches')

# İlk batch'i kontrol et
sample_batch = next(iter(train_loader))
print(f'\nSample batch:')
print(f'  Transaction nodes: {sample_batch["transaction"].num_nodes}')
print(f'  Seed nodes (batch_size): {sample_batch["transaction"].batch_size}')
print(f'  Account nodes: {sample_batch["account"].num_nodes}')
print(f'  Merchant nodes: {sample_batch["merchant"].num_nodes}')
for et in sample_batch.edge_types:
    print(f'  Edge {et}: {sample_batch[et].edge_index.shape[1]}')

Train loader: 21 batches (batch_size=2048)
Val loader: 5 batches

Sample batch:
  Transaction nodes: 14216
  Seed nodes (batch_size): 4096
  Account nodes: 1708
  Merchant nodes: 192
  Edge ('account', 'initiates', 'transaction'): 7856
  Edge ('transaction', 'initiated_by', 'account'): 3856
  Edge ('transaction', 'paid_to', 'merchant'): 480
  Edge ('merchant', 'received_from', 'transaction'): 6154
  Edge ('transaction', 'followed_by', 'transaction'): 4247
  Edge ('transaction', 'preceded_by', 'transaction'): 4336
  Edge ('transaction', 'shares_identity', 'transaction'): 6349
  Edge ('transaction', 'shares_identity_rev', 'transaction'): 6308


In [ ]:
# ─── Differential LR Optimizer ───
lr = cfg['training']['lr']
optimizer = torch.optim.Adam([
    {'params': model.gat.parameters(), 'lr': lr},
    {'params': model.vae.parameters(), 'lr': lr * 0.1},        # 10x düşük
    {'params': model.recon_bn.parameters(), 'lr': lr},
    {'params': model.classifier.parameters(), 'lr': lr * 2},   # 2x yüksek
], weight_decay=cfg['training']['weight_decay'])

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

lambda1 = cfg['training']['lambda1']
lambda2 = cfg['training']['lambda2']
focal_alpha = cfg['training']['focal_alpha']
focal_gamma = cfg['training']['focal_gamma']
max_epochs = cfg['training']['epochs']
patience = cfg['training']['patience']

print('\n' + '═' * 60)
print('PHASE 2: End-to-end training (NeighborLoader)')
print('═' * 60)
print(f'LR: GAT={lr}, VAE={lr*0.1}, Classifier={lr*2}')
print(f'Lambda1(recon)={lambda1}, Lambda2(kl)={lambda2}')
print(f'Focal: alpha={focal_alpha}, gamma={focal_gamma}')

phase2_history = []
best_val_loss = float('inf')
patience_counter = 0
best_state = None
t0 = time.time()

for epoch in range(1, max_epochs + 1):
    # ── TRAIN ──
    model.train()
    train_loss_sum, train_cls_sum, train_recon_sum = 0, 0, 0
    n_train_batches = 0

    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()

        # Seed node sayısı (ilk batch_size transaction'lar seed)
        bs = batch['transaction'].batch_size

        # Forward pass
        raw_txn = batch['transaction'].x  # tüm batch (seed + neighbors)
        outputs = model(batch.x_dict, batch.edge_index_dict, raw_txn_features=raw_txn)

        # Loss sadece seed node'lar üzerinde
        logit = outputs['logit'][:bs]
        raw = outputs['raw_txn'][:bs]
        x_recon = outputs['x_recon'][:bs]
        mu = outputs['mu'][:bs]
        logvar = outputs['logvar'][:bs]
        targets = batch['transaction'].y[:bs].float()

        l_cls = focal_loss(logit, targets, gamma=focal_gamma, alpha=focal_alpha)
        l_recon = reconstruction_loss(raw, x_recon)
        l_kl = kl_divergence(mu, logvar)
        loss = l_cls + lambda1 * l_recon + lambda2 * l_kl

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        train_loss_sum += loss.item()
        train_cls_sum += l_cls.item()
        train_recon_sum += l_recon.item()
        n_train_batches += 1

    avg_train = train_loss_sum / n_train_batches
    avg_cls = train_cls_sum / n_train_batches
    avg_recon = train_recon_sum / n_train_batches

    # ── VALIDATION ──
    model.eval()
    val_loss_sum, n_val_batches = 0, 0

    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(device)
            bs = batch['transaction'].batch_size
            raw_txn = batch['transaction'].x
            outputs = model(batch.x_dict, batch.edge_index_dict, raw_txn_features=raw_txn)

            logit = outputs['logit'][:bs]
            raw = outputs['raw_txn'][:bs]
            x_recon = outputs['x_recon'][:bs]
            mu = outputs['mu'][:bs]
            logvar = outputs['logvar'][:bs]
            targets = batch['transaction'].y[:bs].float()

            l_cls = focal_loss(logit, targets, gamma=focal_gamma, alpha=focal_alpha)
            l_recon = reconstruction_loss(raw, x_recon)
            l_kl = kl_divergence(mu, logvar)
            val_loss = l_cls + lambda1 * l_recon + lambda2 * l_kl

            val_loss_sum += val_loss.item()
            n_val_batches += 1

    avg_val = val_loss_sum / n_val_batches
    scheduler.step(avg_val)

    phase2_history.append({
        'epoch': epoch, 'train_loss': avg_train, 'val_loss': avg_val,
        'cls': avg_cls, 'recon': avg_recon,
    })

    # Early stopping
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        patience_counter = 0
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    else:
        patience_counter += 1

    if epoch % 1 == 0:
        lr_now = optimizer.param_groups[0]['lr']
        print(f'  Epoch {epoch:3d} | cls: {avg_cls:.4f} | recon: {avg_recon:.4f} | val: {avg_val:.4f} | lr: {lr_now:.6f}')

    if patience_counter >= patience:
        print(f'\nEarly stopping at epoch {epoch}')
        break

# Restore best model
if best_state:
    model.load_state_dict(best_state)
    model.to(device)

p2_time = time.time() - t0
print(f'\nPhase 2 tamamlandı ({p2_time:.1f}s). Best val loss: {best_val_loss:.4f}')


════════════════════════════════════════════════════════════
PHASE 2: End-to-end training (NeighborLoader)
════════════════════════════════════════════════════════════
LR: GAT=0.001, VAE=0.0001, Classifier=0.002
Lambda1(recon)=0.3, Lambda2(kl)=0.01
Focal: alpha=0.75, gamma=2.0
  Epoch   1 | cls: 0.0272 | recon: 0.1582 | val: 0.0689 | lr: 0.001000
  Epoch   2 | cls: 0.0216 | recon: 0.1621 | val: 0.0667 | lr: 0.001000
  Epoch   3 | cls: 0.0200 | recon: 0.1619 | val: 0.0661 | lr: 0.001000
  Epoch   4 | cls: 0.0197 | recon: 0.1624 | val: 0.0654 | lr: 0.001000
  Epoch   5 | cls: 0.0190 | recon: 0.1628 | val: 0.0655 | lr: 0.001000
  Epoch   6 | cls: 0.0190 | recon: 0.1624 | val: 0.0651 | lr: 0.001000
  Epoch   7 | cls: 0.0188 | recon: 0.1633 | val: 0.0652 | lr: 0.001000
  Epoch   8 | cls: 0.0187 | recon: 0.1626 | val: 0.0652 | lr: 0.001000
  Epoch   9 | cls: 0.0189 | recon: 0.1629 | val: 0.0646 | lr: 0.001000
  Epoch  10 | cls: 0.0189 | recon: 0.1633 | val: 0.0647 | lr: 0.001000
  Epoch  11

## 6. Test Sonuçları

In [ ]:
# Full-batch evaluation (no_grad → bellek sorunsuz)
model.eval()

# Test için NeighborLoader (tüm komşuları al)
test_loader = NeighborLoader(
    data,
    num_neighbors=[-1, -1],  # tüm komşular (eval için)
    batch_size=4096,
    input_nodes=('transaction', test_mask),
    shuffle=False,
)

all_scores = []
all_targets = []

with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        bs = batch['transaction'].batch_size
        raw_txn = batch['transaction'].x
        outputs = model(batch.x_dict, batch.edge_index_dict, raw_txn_features=raw_txn)
        all_scores.append(outputs['fraud_score'][:bs].cpu())
        all_targets.append(batch['transaction'].y[:bs].cpu())

hybrid_test_scores = torch.cat(all_scores).numpy()
hybrid_test_y = torch.cat(all_targets).numpy()

print('=== Hybrid GAT+VAE (Dual-Branch) ===')
hybrid_metrics = compute_metrics(hybrid_test_y, hybrid_test_scores)
print_report(hybrid_test_y, hybrid_test_scores)

# Score dağılımı
print(f'\nScore istatistikleri:')
print(f'  min: {hybrid_test_scores.min():.4f}, max: {hybrid_test_scores.max():.4f}')
print(f'  mean: {hybrid_test_scores.mean():.4f}, std: {hybrid_test_scores.std():.4f}')
fraud_mask_test = hybrid_test_y == 1
print(f'  fraud mean: {hybrid_test_scores[fraud_mask_test].mean():.4f}')
print(f'  normal mean: {hybrid_test_scores[~fraud_mask_test].mean():.4f}')

## 7. Karşılaştırma Tablosu

In [ ]:
# Sonuç tablosu
results = pd.DataFrame({
    'Model': ['XGBoost', 'GCN Baseline', 'Hybrid GAT+VAE'],
    'F1 (Fraud)': [xgb_metrics['f1_fraud'], gcn_metrics['f1_fraud'], hybrid_metrics['f1_fraud']],
    'Recall': [xgb_metrics['recall_fraud'], gcn_metrics['recall_fraud'], hybrid_metrics['recall_fraud']],
    'Precision': [xgb_metrics['precision_fraud'], gcn_metrics['precision_fraud'], hybrid_metrics['precision_fraud']],
    'AUC-ROC': [xgb_metrics['roc_auc'], gcn_metrics['roc_auc'], hybrid_metrics['roc_auc']],
    'Avg Precision': [xgb_metrics['avg_precision'], gcn_metrics['avg_precision'], hybrid_metrics['avg_precision']],
})
print(results.to_string(index=False))

# Kaydet
os.makedirs('../results/tables', exist_ok=True)
results.to_csv('../results/tables/ieee_cis_results_v2.csv', index=False)

In [ ]:
# Karşılaştırma grafiği
metrics_to_plot = ['F1 (Fraud)', 'Recall', 'Precision', 'AUC-ROC']
x = np.arange(len(metrics_to_plot))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 6))
colors = ['#2ECC71', '#5DADE2', '#E74C3C']
for i, (_, row) in enumerate(results.iterrows()):
    vals = [row[m] for m in metrics_to_plot]
    bars = ax.bar(x + i*width - width, vals, width, label=row['Model'], color=colors[i])
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)

ax.set_ylabel('Score')
ax.set_title('Model Karşılaştırması — IEEE-CIS Dataset')
ax.set_xticks(x)
ax.set_xticklabels(metrics_to_plot)
ax.legend()
ax.set_ylim(0, 1)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()

os.makedirs('../results/figures', exist_ok=True)
plt.savefig('../results/figures/model_comparison_v2.png', dpi=150)
plt.show()

In [ ]:
# Loss eğrileri
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Phase 1
ax1.plot([h['recon'] for h in phase1_history], label='Reconstruction', color='#E74C3C')
ax1.plot([h['kl'] for h in phase1_history], label='KL Divergence', color='#3498DB')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Phase 1: VAE Pre-training (Normal Only)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Phase 2
ax2.plot([h['train_loss'] for h in phase2_history], label='Train Loss', color='#E74C3C')
ax2.plot([h['val_loss'] for h in phase2_history], label='Val Loss', color='#3498DB')
ax2.plot([h['recon'] for h in phase2_history], label='Recon Loss', color='#27AE60', linestyle='--')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.set_title('Phase 2: End-to-End (NeighborLoader)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/figures/training_loss_v2.png', dpi=150)
plt.show()

In [ ]:
# Model kaydet
os.makedirs('../results/models', exist_ok=True)
torch.save(model.state_dict(), '../results/models/hybrid_gatvae_v2.pt')
print('Model kaydedildi: results/models/hybrid_gatvae_v2.pt')

# Eğitim süreleri özeti
print(f'\nEğitim süreleri:')
print(f'  XGBoost:     {xgb_time:.1f}s')
print(f'  GCN:         {gcn_time:.1f}s')
print(f'  Hybrid P1:   {p1_time:.1f}s')
print(f'  Hybrid P2:   {p2_time:.1f}s')
print(f'  Hybrid Total: {p1_time + p2_time:.1f}s')